In [ ]:
print("MCDIA500 • Fase 2: Exploración, Preprocesamiento y Validación • Grupo 5")

MCDIA500 • Fase 2: Exploración, Preprocesamiento y Validación • Grupo 5


# F2 — Exploración, preprocesamiento, transformación y validación del dataset
## Ofertas en licitaciones públicas del sector Salud de marzo de 2026
**Curso:** MCDIA500, Programación para la Ciencia de Datos. **Grupo:** 5.  
**Integrantes:** Víctor Bravo Barrera, Nayadeth Garrido y Alexander Sepulveda.

Esta fase materializa el pipeline de ingeniería y preparación de datos del proyecto, cumpliendo con los estándares de reproducibilidad, modularidad y rigor analítico definidos en la Fase 1:
- **Obtención y trazabilidad:** Carga verificada mediante hash SHA-256 del dataset oficial de ChileCompra.
- **Exploración inicial (EDA):** Diagnóstico de completitud, tipos de datos, cardinalidad y valores faltantes.
- **Limpieza rigurosa:** Exclusión justificada de columnas 100% vacías y neutralización de fechas centinela (año 1900).
- **Transformación de tipos:** Casting de fechas a `datetime64[ns]` y normalización de variables categóricas.
- **Variables derivadas:** Generación de indicadores analíticos (`oferta_ganadora`, `licitacion_adjudicada`, `plazo_cierre_dias`).
- **Validación técnica:** Suite de pruebas automatizadas (casos normales, casos límite y excepciones controladas).
- **Exportación reproducible:** Almacenamiento en `data/processed/` con trazabilidad criptográfica.

## 1. Configuración del entorno y funciones modulares
Se carga el entorno de trabajo y se importan las funciones implementadas en `src/proyecto.py`, garantizando la separación entre lógica de procesamiento y presentación interactiva.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "proyecto.py").is_file() and (p / "F2").is_dir()), None)
if raiz is None:
    raise FileNotFoundError("Abrir el notebook desde la raíz del repositorio o F2.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.proyecto import (
    versiones_entorno, sha256_archivo, leer_datos_f1,
    resumen_exploracion, limpiar_datos_f2, validar_dataset_procesado,
    exportar_datos_procesados
)

print("Directorio raíz del proyecto:", raiz)
display(versiones_entorno())

Directorio raíz del proyecto: C:\Trabajos\sumativo-1


{'numpy': '2.3.5',
 'pandas': '3.0.1',
 'jupyterlab': '4.6.3',
 'ipykernel': '7.3.0',
 'nbformat': '5.11.1',
 'nbconvert': '7.17.1',
 'nbclient': '0.11.0'}

## 2. Obtención y verificación criptográfica de los datos de entrada
Se verifica la huella digital SHA-256 del dataset original `licitaciones_salud_marzo_2026.csv` antes de procesar para asegurar que el insumo no haya sido alterado.

In [ ]:
ruta_raw = raiz / "data" / "raw" / "licitaciones_salud_marzo_2026.csv"
huella_esperada = "490d9209a10d387011d481b72b7891f26e997974ec2cf9dfc518aa4a08552232"
huella_actual = sha256_archivo(ruta_raw)
assert huella_actual == huella_esperada, "Error: la huella SHA-256 del CSV no coincide con la versión documentada."

columnas_esquema = ["NroLicitacion", "TipoLicitacion", "TamanoProveedor", "ResultadoOferta", "EstadoLicitacion", "Sector", "FechaPublicacion"]
df_raw = leer_datos_f1(ruta_raw, columnas_esquema)
print("Dataset de entrada cargado exitosamente:")
print("- Filas totales:", len(df_raw))
print("- Columnas totales:", len(df_raw.columns))
print("- SHA-256 verificado:", huella_actual)
assert df_raw.shape == (44226, 74)

Dataset de entrada cargado exitosamente:
- Filas totales: 44226
- Columnas totales: 74
- SHA-256 verificado: 490d9209a10d387011d481b72b7891f26e997974ec2cf9dfc518aa4a08552232


## 3. Exploración inicial (EDA) y diagnóstico de calidad
Mediante `resumen_exploracion`, analizamos la completitud del conjunto de datos, identificando columnas sin variabilidad o completamente nulas.

In [ ]:
metricas_eda = resumen_exploracion(df_raw)
df_metricas = pd.DataFrame([
    {"Métrica": "Total de filas observadas", "Valor": metricas_eda["total_filas"]},
    {"Métrica": "Total de columnas", "Valor": metricas_eda["total_columnas"]},
    {"Métrica": "Columnas completas (sin nulos)", "Valor": metricas_eda["columnas_completas"]},
    {"Métrica": "Columnas con valores nulos parciales", "Valor": metricas_eda["columnas_con_nulos"]},
    {"Métrica": "Columnas 100% vacías", "Valor": len(metricas_eda["columnas_100_nulos"])},
    {"Métrica": "Columnas con valor único (constantes)", "Valor": len(metricas_eda["columnas_constantes"])},
])
display(df_metricas)
print("Columnas 100% vacías identificadas para exclusión:", metricas_eda["columnas_100_nulos"])
print("Columnas con valor único:", metricas_eda["columnas_constantes"])

,Métrica,Valor
0,Total de filas observadas,44226
1,Total de columnas,74
2,Columnas completas (sin nulos),49
3,Columnas con valores nulos parciales,22
4,Columnas 100% vacías,3
5,Columnas con valor único (constantes),7


Columnas 100% vacías identificadas para exclusión: ['LicitacionBaseTipo', 'ContratoRenovable', 'UnidadTiempoRenovacion']
Columnas con valor único: ['LicitacionInformada', 'LicitacionBaseTipo', 'TipoAdjudicacion', 'ContratoRenovable', 'ValorTiempoRenovacion', 'UnidadTiempoRenovacion', 'Sector']


### Distribuciones de las variables clave del estudio
Examinamos la distribución de frecuencias de las variables principales (`TipoLicitacion`, `TamanoProveedor`, `ResultadoOferta`) y su interacción con `EstadoLicitacion`.

In [ ]:
print("1. Distribución por Tipo de Licitación:")
display(df_raw["TipoLicitacion"].value_counts().to_frame("Frecuencia"))

print("2. Distribución por Tamaño del Proveedor:")
display(df_raw["TamanoProveedor"].value_counts().to_frame("Frecuencia"))

print("3. Distribución por Resultado de la Oferta:")
display(df_raw["ResultadoOferta"].value_counts().to_frame("Frecuencia"))

print("4. Matriz de contingencia: EstadoLicitacion vs ResultadoOferta:")
display(pd.crosstab(df_raw["EstadoLicitacion"], df_raw["ResultadoOferta"], margins=True))

1. Distribución por Tipo de Licitación:


,Frecuencia
TipoLicitacion,
Licitación Pública Entre 100 y 1000 UTM (LE),23708
Licitación Pública Mayor 1000 UTM (LP),14448
Licitación Pública Mayor a 5000 (LR),3227
Licitación Pública Menor a 100 UTM (L1),2757
Licitación Privada Mayor a 1000 UTM,72
Licitación Privada entre 100 y 1000 UTM.,12
Licitación Privada Mayor a 5000 (I2),2


2. Distribución por Tamaño del Proveedor:


,Frecuencia
TamanoProveedor,
Grande,18865
Pequeña,9682
Mediana,9015
NoClasificado,3410
Micro,3254


3. Distribución por Resultado de la Oferta:


,Frecuencia
ResultadoOferta,
Ganadora,26063
Perdedora,18163


4. Matriz de contingencia: EstadoLicitacion vs ResultadoOferta:


ResultadoOferta,Ganadora,Perdedora,All
EstadoLicitacion,,,
Adjudicada,26062,17963,44025
Cerrada,1,55,56
Desierta (o art. 3 ó 9 Ley 19.886),0,138,138
Revocada,0,7,7
All,26063,18163,44226


## 4. Pipeline modular de limpieza y transformación de datos
Se ejecuta la función `limpiar_datos_f2(df_raw)` de `src/proyecto.py`. Sus etapas y justificaciones son:
1. **Exclusión de columnas 100% vacías:** Se descartan `LicitacionBaseTipo`, `ContratoRenovable` y `UnidadTiempoRenovacion`.
2. **Neutralización de fechas centinela:** Los 7.613 valores de `FechaEstimadaEvaluacionOfertas` fijados en el año 1900 corresponden a valores por defecto del sistema de origen; se convierten a `NaT`.
3. **Casting a datetime:** Las columnas de fecha se convierten a formato de fecha/hora de pandas (`datetime64[ns]`), permitiendo cálculos de plazos.
4. **Normalización de texto:** Se remueven espacios en blanco residuales (`str.strip()`).
5. **Preservación de categorías:** Se preserva la etiqueta `NoClasificado` en `TamanoProveedor` (3.410 ofertas), evitando imputaciones artificiales.
6. **Ingeniería de variables:**
   - `oferta_ganadora`: booleano (`True` si `ResultadoOferta == 'Ganadora'`).
   - `licitacion_adjudicada`: booleano (`True` si `EstadoLicitacion == 'Adjudicada'`).
   - `plazo_cierre_dias`: cálculo de días entre publicación y cierre del proceso.

In [6]:
df_procesado = limpiar_datos_f2(df_raw)
print("Dimensiones tras preprocesamiento:", df_procesado.shape)

cols_excluidas = ["LicitacionBaseTipo", "ContratoRenovable", "UnidadTiempoRenovacion"]
presentes = [c for c in cols_excluidas if c in df_procesado.columns]
assert len(presentes) == 0, f"Error: columnas vacías aún presentes: {presentes}"
print("OK: Columnas vacías excluidas correctamente.")

print("Tipo inferido FechaPublicacion:", df_procesado["FechaPublicacion"].dtype)
print("Tipo inferido FechaCierre:", df_procesado["FechaCierre"].dtype)
print("Tipo inferido FechaAdjudicacion:", df_procesado["FechaAdjudicacion"].dtype)

plazo_stats = df_procesado["plazo_cierre_dias"].describe()
print("\nEstadísticos de plazo_cierre_dias:")
display(plazo_stats.to_frame())

Dimensiones tras preprocesamiento: (44226, 74)
OK: Columnas vacías excluidas correctamente.
Tipo inferido FechaPublicacion: datetime64[us]
Tipo inferido FechaCierre: datetime64[us]
Tipo inferido FechaAdjudicacion: datetime64[us]

Estadísticos de plazo_cierre_dias:


,plazo_cierre_dias
count,44226.000000
mean,14.384128
std,6.862530
min,4.917241
25%,10.082613
50%,12.035603
75%,19.987572
max,61.032607


## 5. Análisis exploratorio de proporciones de éxito
Con los datos limpios, comparamos la tasa de adjudicación según el tamaño del proveedor en licitaciones que culminaron efectivamente en adjudicación (`licitacion_adjudicada == True`), evitando distorsiones por procesos desiertos o cancelados.

In [7]:
adj = df_procesado[df_procesado["licitacion_adjudicada"]]
conteo_tamano = pd.crosstab(adj["TamanoProveedor"], adj["ResultadoOferta"], margins=True)
pct_tamano = (pd.crosstab(adj["TamanoProveedor"], adj["ResultadoOferta"], normalize="index") * 100).round(2)
pct_tamano.columns = [f"% {c}" for c in pct_tamano.columns]

tabla_resumen_tamano = conteo_tamano.join(pct_tamano)
print("Tasa de éxito de ofertas por tamaño de proveedor (Licitaciones Adjudicadas):")
display(tabla_resumen_tamano)

Tasa de éxito de ofertas por tamaño de proveedor (Licitaciones Adjudicadas):


,Ganadora,Perdedora,All,% Ganadora,% Perdedora
TamanoProveedor,,,,,
Grande,12575,6255,18830,66.78,33.22
Mediana,4982,4001,8983,55.46,44.54
Micro,1680,1554,3234,51.95,48.05
NoClasificado,1803,1576,3379,53.36,46.64
Pequeña,5022,4577,9599,52.32,47.68
All,26062,17963,44025,NaN,NaN


## 6. Validación técnica y verificación del código
Para satisfacer el criterio de *Validación técnica y verificación del código*, se ejecutan pruebas automáticas que comprueban el comportamiento del pipeline en:
1. **Caso Normal:** Verificación integral de 6 reglas de calidad sobre el dataset real mediante `validar_dataset_procesado`.
2. **Caso Límite (Edge Case):** Comportamiento frente a fechas centinela 1900 y registros con proveedor `NoClasificado`.
3. **Caso Excepción 1 (Columnas ausentes):** Intento de procesar un dataset sin columna obligatoria, verificando captura de `KeyError`.
4. **Caso Excepción 2 (Dataset vacío):** Intento de procesar un DataFrame sin registros, verificando captura de `ValueError`.

In [8]:
pruebas_f2 = []

# 1. Caso Normal
res_val = validar_dataset_procesado(df_procesado)
pruebas_f2.append({
    "Caso": "Caso Normal (Dataset Real)",
    "Tipo": "Validación integral",
    "Resultado": res_val["estado"],
    "Detalle": f"{res_val['reglas_superadas']} reglas superadas sobre {res_val['filas_validadas']:,} registros"
})

# 2. Caso Límite
df_limite = pd.DataFrame({
    "NroLicitacion": ["LIC-TEST-01", "LIC-TEST-02"],
    "TipoLicitacion": ["LE", "LP"],
    "TamanoProveedor": ["NoClasificado", "Micro"],
    "ResultadoOferta": ["Ganadora", "Perdedora"],
    "EstadoLicitacion": ["Adjudicada", "Adjudicada"],
    "FechaPublicacion": ["2026-03-01 10:00:00.000", "2026-03-02 12:00:00.000"],
    "FechaCierre": ["2026-03-15 18:00:00.000", "2026-03-20 18:00:00.000"],
    "FechaEstimadaEvaluacionOfertas": ["1900-01-01 00:00:00.000", np.nan]
})
df_limite_proc = limpiar_datos_f2(df_limite)
assert df_limite_proc["FechaEstimadaEvaluacionOfertas"].isna().all()
assert df_limite_proc.loc[0, "TamanoProveedor"] == "NoClasificado"
pruebas_f2.append({
    "Caso": "Caso Límite (Fechas 1900 y NoClasificado)",
    "Tipo": "Valores frontera / centinelas",
    "Resultado": "OK",
    "Detalle": "Fecha 1900 neutralizada a NaT y categoría NoClasificado preservada intacta"
})

# 3. Caso Excepción: Columna obligatoria ausente
df_sin_resultado = df_limite.drop(columns=["ResultadoOferta"])
try:
    limpiar_datos_f2(df_sin_resultado)
except KeyError as err:
    pruebas_f2.append({
        "Caso": "Caso Excepción: Columna faltante",
        "Tipo": "Manejo de errores de esquema",
        "Resultado": "OK",
        "Detalle": f"Captura controlada de KeyError ({err})"
    })
else:
    raise AssertionError("Fallo en prueba: no se detectó la columna ausente.")

# 4. Caso Excepción: Dataset sin filas
df_vacio = pd.DataFrame(columns=df_limite.columns)
try:
    limpiar_datos_f2(df_vacio)
except ValueError as err:
    pruebas_f2.append({
        "Caso": "Caso Excepción: DataFrame vacío",
        "Tipo": "Manejo de conjuntos vacíos",
        "Resultado": "OK",
        "Detalle": f"Captura controlada de ValueError ({err})"
    })
else:
    raise AssertionError("Fallo en prueba: no se detectó el DataFrame vacío.")

display(pd.DataFrame(pruebas_f2))
print("OK: Todas las pruebas de verificación del código fueron superadas exitosamente.")

,Caso,Tipo,Resultado,Detalle
0,Caso Normal (Dataset Real),Validación integral,OK,"6 reglas superadas sobre 44,226 registros"
1,Caso Límite (Fechas 1900 y NoClasificado),Valores frontera / centinelas,OK,Fecha 1900 neutralizada a NaT y categoría NoCl...
2,Caso Excepción: Columna faltante,Manejo de errores de esquema,OK,"Captura controlada de KeyError (""Faltan column..."
3,Caso Excepción: DataFrame vacío,Manejo de conjuntos vacíos,OK,Captura controlada de ValueError (No es posibl...


OK: Todas las pruebas de verificación del código fueron superadas exitosamente.


## 7. Exportación del dataset procesado y registro de trazabilidad
El conjunto procesado se almacena en `data/processed/licitaciones_salud_marzo_2026_procesado.csv` y se computa su huella SHA-256 para permitir trazabilidad absoluta en las siguientes entregas.

In [9]:
ruta_salida = raiz / "data" / "processed" / "licitaciones_salud_marzo_2026_procesado.csv"
info_exp = exportar_datos_procesados(df_procesado, ruta_salida)
display(pd.DataFrame([info_exp]))
print("Dataset exportado exitosamente a:", info_exp["ruta"])
print(f"Dimensiones exportadas: {info_exp['filas']:,} filas x {info_exp['columnas']} columnas ({info_exp['tamanio_mb']} MB)")
print("SHA-256 verificado:", info_exp["sha256"])

,archivo,ruta,filas,columnas,tamanio_mb,sha256
0,licitaciones_salud_marzo_2026_procesado.csv,C:\Trabajos\sumativo-1\data\processed\licitaci...,44226,74,68.16,7e835d6725d8c4f9aa64ad97a294dc38fcc341e71373f5...


Dataset exportado exitosamente a: C:\Trabajos\sumativo-1\data\processed\licitaciones_salud_marzo_2026_procesado.csv
Dimensiones exportadas: 44,226 filas x 74 columnas (68.16 MB)
SHA-256 verificado: 7e835d6725d8c4f9aa64ad97a294dc38fcc341e71373f544c7f6b8ba2ff31dc5


## 8. Hallazgos técnicos y articulación con Fases 3 y 4
1. **Calidad de datos y trazabilidad:** Se validó la integridad de las 44.226 observaciones de ofertas del sector Salud. El tratamiento sistemático de nulos y fechas centinela garantiza que los análisis posteriores no contengan sesgos instrumentales.
2. **Patrones observados:** Las empresas catalogadas como **Grande** presentan una tasa de éxito de adjudicación (66.78%) significativamente superior a las **Medianas** (55.46%), **Pequeñas** (52.32%) y **Micro** (51.95%).
3. **Próximos pasos (F3 y F4):** En la Fase 3 se desarrollarán visualizaciones de distribuciones y pruebas estadísticas para contrastar si estas diferencias son estadísticamente significativas por tipo de licitación.

## 9. Referencias bibliográficas (APA 7.ª edición)
- ChileCompra. (s. f.). *Datos abiertos: Descargas*. Recuperado el 12 de septiembre de 2026, de https://datos-abiertos.chilecompra.cl/descargas
- McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3.ª ed.). O'Reilly Media.
- NumPy Developers. (s. f.). *NumPy documentation*. Recuperado el 12 de septiembre de 2026, de https://numpy.org/doc/stable/
- Samuel, S., & Mietchen, D. (2024). Computational reproducibility of Jupyter notebooks from biomedical publications. *GigaScience*, 13, giad113. https://doi.org/10.1093/gigascience/giad113
- The pandas development team. (s. f.). *pandas documentation*. Recuperado el 12 de septiembre de 2026, de https://pandas.pydata.org/docs/